# Anomaly-scoring validation
Seeded synthetic injections: compare within-dataset forest ranking with tests-only flag count. Scores are relative, not probabilities. Results will vary by scenario; no fixed superiority is assumed.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from fraudscope.runner import Config
from fraudscope.service import run_analysis

def evaluate(seed, kind, n=600):
    rng = np.random.default_rng(seed)
    clean = 10 ** rng.uniform(1, 6, n)
    injected = clean.copy()
    labels = np.zeros(n, dtype=int)
    labels[:n//6] = 1
    m = n//6
    if kind == 'digit_preference': injected[:m] = 55500
    elif kind == 'rounding': injected[:m] = np.round(injected[:m] / 100) * 100
    elif kind == 'uniform_digits': injected[:m] = rng.integers(1, 10, m) * 10 ** rng.integers(2, 6, m)
    elif kind == 'duplicates': injected[:m] = injected[0]
    elif kind == 'scaled_subset': injected[:m] *= 5
    elif kind == 'regression_outliers': injected[:m] *= 10
    config = Config(['amount'], ['first', 'last_two'], references={'amount': 'benford'}, random_state=seed, n_sim=200)
    output = run_analysis(pd.DataFrame({'amount': injected}), config)
    clean_output = run_analysis(pd.DataFrame({'amount': clean}), config)
    scores = output.ml_output.scores
    baseline = output.ml_output.baseline
    precision = lambda values: labels[np.argsort(-np.asarray(values))[:m]].mean()
    return dict(seed=seed, injection=kind, forest_auc=roc_auc_score(labels, scores), baseline_auc=roc_auc_score(labels, baseline), forest_precision_at_k=precision(scores), baseline_precision_at_k=precision(baseline), clean_index=clean_output.ml_output.index, injected_index=output.ml_output.index)

cases = ['digit_preference', 'rounding', 'uniform_digits', 'duplicates', 'scaled_subset', 'regression_outliers']
validation = pd.DataFrame([evaluate(seed, kind) for seed in (2, 7, 17) for kind in cases])
validation.groupby('injection').mean(numeric_only=True).round(3)


Check where the forest underperforms the tests-only baseline. Scaling a subset can preserve leading digit distributions; `regression_outliers` above perturbs amounts without a fitted regression. The next cell checks actual fitted regression outliers. Clean and injected indices are descriptive and not guaranteed to separate for every injection.

In [ ]:
validation.assign(auc_gain=validation.forest_auc-validation.baseline_auc, precision_gain=validation.forest_precision_at_k-validation.baseline_precision_at_k).round(3)


In [ ]:
from fraudscope.runner import RegressionSpec

def regression_case(seed, n=600):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=n)
    response = 2 + 3*x + rng.normal(scale=.5, size=n)
    labels = np.zeros(n, dtype=int)
    labels[:n//6] = 1
    clean = pd.DataFrame({'amount': 10 ** rng.uniform(1, 6, n), 'predictor': x, 'response': response})
    injected = clean.copy()
    injected.loc[:n//6-1, 'response'] += 12
    config = Config(['amount'], regression=RegressionSpec('response', ['predictor']), random_state=seed)
    output = run_analysis(injected, config).ml_output
    clean_index = run_analysis(clean, config).ml_output.index
    precision = lambda scores: labels[np.argsort(-np.asarray(scores))[:n//6]].mean()
    return {'seed': seed, 'forest_auc': roc_auc_score(labels, output.scores), 'baseline_auc': roc_auc_score(labels, output.baseline), 'forest_precision_at_k': precision(output.scores), 'baseline_precision_at_k': precision(output.baseline), 'clean_index': clean_index, 'injected_index': output.index}

pd.DataFrame([regression_case(seed) for seed in (2, 7, 17)]).mean().round(3)
